# LangChain Indexing API Reference

Developer-facing statements defined in `langchain_core.indexing.api`.

# `IndexingException: LangChainException`

Raised when an indexing operation fails.

The indexing functions raise this exception when a destination deletion reports failure, including a `VectorStore` returning `False` or a `DocumentIndex` reporting failed deletions.

---

# `IndexingResult: TypedDict`

Detailed counts returned by an indexing operation.

```python
num_added: int # Number of newly added documents
num_updated: int # Number of existing documents updated because force_update was enabled
num_deleted: int # Number of documents deleted during cleanup
num_skipped: int # Number of duplicate or already-current documents skipped
```

---

# `index`

Indexes documents into a vector store or document index while using a record manager to track additions, updates, skips, and deletions.

```python
index(
    docs_source: BaseLoader | Iterable[Document], # Loader or iterable supplying documents
    record_manager: RecordManager, # Timestamped record manager tracking indexed document IDs
    vector_store: VectorStore | DocumentIndex, # Destination receiving document upserts and deletions
    *,
    batch_size: int = 100, # Number of documents processed per indexing batch
    cleanup: Literal["incremental", "full", "scoped_full"] | None = None, # Cleanup strategy
    source_id_key: str | Callable[[Document], str] | None = None, # Metadata key or callable assigning source IDs
    cleanup_batch_size: int = 1_000, # Maximum number of records deleted per cleanup request
    force_update: bool = False, # Whether existing documents should be written again
    key_encoder: Literal["sha1", "sha256", "sha512", "blake2b"] | Callable[[Document], str] = "sha1", # Hash algorithm or custom document ID encoder
    upsert_kwargs: dict[str, Any] | None = None, # Extra arguments forwarded to the destination write method
) -> IndexingResult # Counts of added, updated, deleted, and skipped documents
```

## Behaviour

Documents receive deterministic IDs generated from their content and JSON-serialized metadata, unless `key_encoder` is a callable. Documents with the same generated ID inside one batch are deduplicated while preserving order.

The default `"sha1"` encoder emits a one-time `UserWarning` because SHA-1 is not collision-resistant. The source marks `key_encoder` as added in `langchain-core` 0.3.66. Changing the encoder requires changing the index to avoid duplicate records.

For a `BaseLoader`, `lazy_load()` is preferred and `load()` is used when lazy loading raises `NotImplementedError`.

Destination writes occur before the record manager is updated. Existing records have their timestamps refreshed; with `force_update=True`, they are written again and counted as updated.

Cleanup modes behave as follows:

* `"incremental"` continuously deletes stale records associated with source IDs encountered in each batch.
* `"full"` deletes every stale record after all input documents have been processed.
* `"scoped_full"` deletes stale records only for source IDs encountered during the run and keeps those source IDs in memory.
* `None` performs no cleanup.

`"scoped_full"` was added in `langchain-core` 0.3.25. `upsert_kwargs`, added in `langchain-core` 0.3.10, is forwarded to `VectorStore.add_documents()` or `DocumentIndex.upsert()`.

## Exceptions

Raises `ValueError` for an unsupported cleanup mode, a non-positive indexing batch size, a missing `source_id_key` for `"incremental"` or `"scoped_full"` cleanup, a document without an assigned source ID in those modes, an invalid source-ID assigner, unsupported hashing, non-JSON-serializable metadata, or a `VectorStore` missing the required write or delete implementation.

Raises `TypeError` when `vector_store` is neither a `VectorStore` nor a `DocumentIndex`.

Raises `IndexingException` when destination deletion reports failure.

An `AssertionError` protects an unreachable state in which a source ID becomes `None` during incremental cleanup.

---

# `aindex`

Asynchronously indexes documents while using asynchronous record-manager and destination operations.

```python
async aindex(
    docs_source: BaseLoader | Iterable[Document] | AsyncIterator[Document], # Loader, synchronous iterable, or asynchronous document iterator
    record_manager: RecordManager, # Record manager accessed through its asynchronous methods
    vector_store: VectorStore | DocumentIndex, # Asynchronous indexing destination
    *,
    batch_size: int = 100, # Number of documents processed per asynchronous batch
    cleanup: Literal["incremental", "full", "scoped_full"] | None = None, # Cleanup strategy matching index()
    source_id_key: str | Callable[[Document], str] | None = None, # Metadata key or callable producing source IDs
    cleanup_batch_size: int = 1_000, # Maximum number of records deleted in each cleanup operation
    force_update: bool = False, # Whether existing documents should be upserted again
    key_encoder: Literal["sha1", "sha256", "sha512", "blake2b"] | Callable[[Document], str] = "sha1", # Built-in hash algorithm or custom encoder
    upsert_kwargs: dict[str, Any] | None = None, # Extra destination upsert arguments
) -> IndexingResult # Asynchronous indexing counts
```

## Behaviour

This function applies the same hashing, deduplication, timestamp-refresh, force-update, cleanup, and result-counting rules as `index()`.

For a `BaseLoader`, it uses `alazy_load()`. When asynchronous lazy loading raises `NotImplementedError`, it calls `load()` and exposes the returned documents through an asynchronous iterator. A synchronous iterable is also converted to an asynchronous iterator.

It writes through `VectorStore.aadd_documents()` or `DocumentIndex.aupsert()` and deletes through the corresponding asynchronous destination operation before updating the record manager.

## Exceptions

Raises the same cleanup, hashing, metadata, source-ID, destination-type, deletion-failure, and unreachable-state exceptions as `index()`.

For a `VectorStore`, it additionally raises `ValueError` when neither asynchronous nor synchronous deletion is implemented, or when the required asynchronous write method is unavailable.

In [ ]:
from langchain_core.documents import Document # Import the Document class
from langchain_core.indexing import InMemoryRecordManager, index # Import indexing tools
from langchain_core.indexing.in_memory import InMemoryDocumentIndex # Import an in-memory document index

documents = [ # Create documents to index
    Document(page_content="Python is a programming language.", metadata={"source": "python"}), # Create the first document
    Document(page_content="LangChain helps build LLM applications.", metadata={"source": "langchain"}), # Create the second document
] # Finish the document list

record_manager = InMemoryRecordManager(namespace="demo") # Track indexed documents
document_index = InMemoryDocumentIndex() # Create an in-memory destination

first_result = index( # Run the first indexing operation
    documents, # Provide the documents
    record_manager, # Provide the record manager
    document_index, # Provide the destination
    cleanup="incremental", # Remove outdated documents from the same source
    source_id_key="source", # Read source IDs from metadata
    key_encoder="sha256", # Generate stable document IDs
) # Finish the first operation

second_result = index( # Index the same documents again
    documents, # Provide the unchanged documents
    record_manager, # Reuse the record manager
    document_index, # Reuse the destination
    cleanup="incremental", # Use the same cleanup strategy
    source_id_key="source", # Use the same source field
    key_encoder="sha256", # Use the same ID encoder
) # Finish the second operation

print("First indexing:", first_result) # Show that two documents were added
print("Second indexing:", second_result) # Show that two documents were skipped
print("Stored documents:", len(document_index.store)) # Show the stored document count